In [1]:
import os

import cartopy.crs as ccrs
import cmcrameri as cmc  # noqa: F401
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import yaml
from matplotlib.gridspec import GridSpec
from metpy.units import units
from unseen_awg.plotting_utils import map_plot_without_frame_with_bounds

In [2]:
mpl.rc_file("../../matplotlibrc")

In [3]:
with open("../../configs/paths.yaml") as file:
    paths = yaml.safe_load(file)["paths"]

In [5]:
reforecast_impacts = xr.open_zarr(
    os.path.join(
        paths["dir_preprocessed_datasets"],
        "preprocessed_impact_variables_reforecasts",
        "combined-corrected_facc0e91_7d1d3d97_884a804a_3f7e331d.zarr",
    )
)

reforecast_impacts["tp"] = (
    reforecast_impacts["tp"]
    .where(reforecast_impacts["tp"].metpy.quantify() > 1 * units.millimeter, 0)
    .metpy.dequantify()
)

rng = np.random.default_rng(seed=0)

n_datapoints = 5

random_ensemble_members = xr.DataArray(
    rng.choice(reforecast_impacts.ensemble_member, size=n_datapoints),
    dims="datapoint",
)
random_lead_times = xr.DataArray(
    rng.choice(reforecast_impacts.lead_time, size=n_datapoints), dims="datapoint"
)
random_init_times = xr.DataArray(
    rng.choice(reforecast_impacts.init_time, size=n_datapoints), dims="datapoint"
)

data = reforecast_impacts.sel(
    ensemble_member=random_ensemble_members,
    lead_time=random_lead_times,
    init_time=random_init_times,
).load()

In [6]:
cmaps = {
    "t2m": {
        "cmap": plt.get_cmap("RdBu_r"),
        "norm": mpl.colors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40),
        "cbar_kwargs": {"label": r"$T_{2m, mean}$ [°C]", "orientation": "horizontal"},
    },
    "mn2t": {
        "cmap": plt.get_cmap("RdBu_r"),
        "norm": mpl.colors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40),
        "cbar_kwargs": {"label": r"$T_{2m, min}$ [°C]", "orientation": "horizontal"},
    },
    "mx2t": {
        "cmap": plt.get_cmap("RdBu_r"),
        "norm": mpl.colors.TwoSlopeNorm(vmin=-20, vcenter=0, vmax=40),
        "cbar_kwargs": {"label": r"$T_{2m, max}$ [°C]", "orientation": "horizontal"},
    },
    "tp": {
        "cmap": plt.get_cmap("Blues"),
        "norm": mpl.colors.LogNorm(vmin=1, vmax=40),
        "cbar_kwargs": {"label": r"$tp$ [mm]", "orientation": "horizontal"},
    },
}
cmaps["tp"]["cmap"].set_bad("white")

In [ ]:
path_images = os.path.join(paths["dir_images"], "example_states")
os.makedirs(path_images, exist_ok=True)

for dp in data.datapoint:
    for var in ["t2m", "tp"]:
        fig = plt.figure()
        gs = GridSpec(1, 1)

        font_kwargs = dict(fontweight="bold", fontsize="large")

        ax_base = fig.add_subplot(gs[0, 0], projection=ccrs.Robinson())
        map_plot_without_frame_with_bounds(
            ax=ax_base,
            da=data.sel(datapoint=dp)[var].squeeze(),
            cmap=cmaps[var]["cmap"],
            norm=cmaps[var]["norm"],
            add_colorbar=False,
            rasterized=True,
        )
        ax_base.set_title("")

        plt.savefig(os.path.join(path_images, f"{var}_{dp.data}.svg"))
